In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import matplotlib.pyplot as plot
import seaborn as sns

from glob import glob

import librosa
import librosa.display
import IPython.display as ipd

from itertools import cycle


color_pal = plt.rcParams["axes.prop_cycle"].by_key()["color"]
color_cycle = cycle(plt.rcParams["axes.prop_cycle"].by_key()["color"])

### experimental personal imports
helping map points with tkinter using twodpm (two dimensional)
help map 3d coordinates using threedpm (three dimensional)


In [ ]:
import src.tkinter_point_mapper as td
from importlib import reload
reload(td)

In [ ]:
audio_files = glob("./*.mp3")
other_files = glob("music/*.mp3")
audio_files.extend(other_files)
audio_files



In [ ]:
working_file = audio_files[4]
ipd.Audio(working_file)

In [ ]:
y, sr = librosa.load(working_file)
print(f'y: {y[:10]}')
print(f'shape y: {y.shape}')
print(f'sr: {sr}')
# in one second of audio there are {sr} samples
# length of audio in seconds = len(y) / sr
# to get a any stretch of time, second*sr : (second+duration)*sr
d = librosa.amplitude_to_db(y)

In [ ]:
#Displaying the audio (to find high and low deciblel)
pd.Series(d).plot(figsize=(10, 5),
                  lw=1,
                  title='Raw Audio display',
                 color=color_pal[0])
plt.show()

In [ ]:
y, _ = librosa.effects.trim(y, top_db=20)

# Isolating vocals with librosa

### Loading file

In [ ]:
all_data, sr = librosa.load('orion-sun-05-2.mp3')
#all_data, sr = librosa.load('music/new david bowie.mp3')


# And compute the spectrogram magnitude and phase
S_full, phase = librosa.magphase(librosa.stft(all_data))

# Play back a 5-second excerpt with vocals
ipd.Audio(data=all_data[10*sr:15*sr], rate=sr)

### Change these values to adjust vocal / background level

margin_i : background 

margin_v : foreground

power : 

In [ ]:
# We can also use a margin to reduce bleed between the vocals and instrumentation masks.
# Note: the margins need not be equal for foreground and background separation
margin_i, margin_v = 5, 2
power = 4

### Isolate vocals

credits: https://librosa.org/doc/0.9.2/auto_examples/plot_vocal_separation.html

In [ ]:
#plotting 
idx = slice(*librosa.time_to_frames([10, 20], sr=sr))
fig, ax = plt.subplots()
img = librosa.display.specshow(librosa.amplitude_to_db(S_full[:, idx], ref=np.max),
                         y_axis='log', x_axis='time', sr=sr, ax=ax)
fig.colorbar(img, ax=ax)


S_filter = librosa.decompose.nn_filter(S_full,
                                       aggregate=np.median,
                                       metric='cosine',
                                       width=int(librosa.time_to_frames(2, sr=sr)))

# The output of the filter shouldn't be greater than the input
# if we assume signals are additive.  Taking the pointwise minimum
# with the input spectrum forces this.
S_filter = np.minimum(S_full, S_filter)



mask_i = librosa.util.softmask(S_filter,
                               margin_i * (S_full - S_filter),
                               power=power)

mask_v = librosa.util.softmask(S_full - S_filter,
                               margin_v * S_filter,
                               power=power)

# Once we have the masks, simply multiply them with the input spectrum
# to separate the components

S_foreground = mask_v * S_full
S_background = mask_i * S_full

#plotting seperate
fig, ax = plt.subplots(nrows=3, sharex=True, sharey=True)
img = librosa.display.specshow(librosa.amplitude_to_db(S_full[:, idx], ref=np.max),
                         y_axis='log', x_axis='time', sr=sr, ax=ax[0])
ax[0].set(title='Full spectrum')
ax[0].label_outer()

librosa.display.specshow(librosa.amplitude_to_db(S_background[:, idx], ref=np.max),
                         y_axis='log', x_axis='time', sr=sr, ax=ax[1])
ax[1].set(title='Background')
ax[1].label_outer()

librosa.display.specshow(librosa.amplitude_to_db(S_foreground[:, idx], ref=np.max),
                         y_axis='log', x_axis='time', sr=sr, ax=ax[2])
ax[2].set(title='Foreground')
fig.colorbar(img, ax=ax)



y_foreground = librosa.istft(S_foreground * phase)
y_background = librosa.istft(S_background * phase)
# Play back a 5-second excerpt with only vocals
# ISOLATED VOCALS
ipd.Audio(data=y_foreground, rate=sr)

#### Background track

In [ ]:
ipd.Audio(data=y_background, rate=sr)

# Make spectrographs

In [ ]:
audio_file, sample_rate = librosa.load(audio_files[1])
melspectrogram_of_audio = librosa.feature.melspectrogram(y=audio_file, sr=sample_rate)
ipd.Audio(data=audio_file, rate=sample_rate)



In [ ]:
n_fft = 2048 #optimal value for music
print(f'sample_rate: {sample_rate}')
hop_length = 22050 // 4  # 1/4 second hop for sample rate of 22050



power_spectrogram = np.abs(librosa.stft(audio_file, n_fft=n_fft, hop_length=hop_length, center=True))
freqs = librosa.fft_frequencies(sr=sample_rate, n_fft=n_fft)
times = librosa.frames_to_time(power_spectrogram[0], sr=sample_rate, hop_length=hop_length)
time_sum_array = np.cumsum(times)
print(f'power spectrogram shape: {power_spectrogram.shape}')




fft_bin = 1 # max value is 1 + n_fft/2
time_idx = 4

print('freq (Hz)', freqs[fft_bin])
print(f'playtime:{time_idx*(hop_length/sample_rate)} seconds. time=idx: {times[time_idx]}')
print(f'amplitude at frequency bin {fft_bin} and time {time_idx*(hop_length/sample_rate)}: {power_spectrogram[fft_bin, time_idx]}')
print('decibels', librosa.amplitude_to_db(power_spectrogram[fft_bin, time_idx]))
print(freqs.shape)

In [ ]:
#splitting frequency 
min_freq = 20
max_freq = 20000

final_decibels_per_frequency = []

#loop through all indexes in times?
decibel_at_freqtime = []
for i in range(len(freqs)):
    
    if freqs[i] < min_freq or freqs[i] > max_freq:
        print(f'skipping frequency: {freqs[i]}')
        continue
    # process frequency
    #print(f'processing frequency: {freqs[i]}')
    #get index
    decibel_at_freqtime.append(power_spectrogram[i,100])


decibel_at_freqtime







## Generating data

In [ ]:
print(audio_files)
ipd.Audio(audio_files[2])


In [ ]:
audio_file, sample_rate = librosa.load(audio_files[2])
second_per_data = 10 #change here for new sample rate
n_fft = 2048 #optimal value for music is 2049 - number of frequency buckets = n_ffft/2 + 1
print(f'sample_rate: {sample_rate}')
hop_length = int(sample_rate // 24) # sample rate/x for x frames per second

power_spectrogram = np.abs(librosa.stft(audio_file, n_fft=n_fft, hop_length=hop_length, center=True))
freqs = librosa.fft_frequencies(sr=sample_rate, n_fft=n_fft)
times = librosa.frames_to_time(power_spectrogram[0], sr=sample_rate, hop_length=hop_length)
time_sum_array = np.cumsum(times)
print(f'power spectrogram shape: {power_spectrogram.shape}')
times = [i*hop_length/sample_rate for i in range(power_spectrogram.shape[1])]
print(f'last time: {times[-1]}')

spectrogram_table = pd.DataFrame(power_spectrogram)
print(f'frequency length: {len(freqs)}, time length:{len(times)}')

# Setting labels to the exact time and exact frequency, if want to work in terms of frames (instead of time) and frequency/index
#spectrogram_table = spectrogram_table.set_axis(times, axis=1)
#spectrogram_table = spectrogram_table.set_axis(freqs, axis=0)

print(spectrogram_table)





In [ ]:
## Save data to csv
## rows as frequencies, columns as time
spectrogram_table.to_csv('spectrogram_data.csv', index=True)

## Quickly visualise points

In [ ]:
column = 2
print(spectrogram_table[spectrogram_table.columns[column]].index[2])
#changing one row (frequency data) to x,y coodinates
#want x value to be time and y value to be decibel/value
coordinates = []
for i in range(len(spectrogram_table[spectrogram_table.columns[column]])):
    x = spectrogram_table[spectrogram_table.columns[column]].index[i]
    y = spectrogram_table.iloc[i, column]
    coordinates.append((float(x), float(y)))

coordinates[0:5]

timeseries_coords = []
for columnnum in spectrogram_table.columns:
    newcoordinates = []
    for i in range(len(spectrogram_table[spectrogram_table.columns[columnnum]])):
        x = spectrogram_table[spectrogram_table.columns[columnnum]].index[i]
        y = spectrogram_table.iloc[i, columnnum]
        newcoordinates.append((float(x), float(y)))
    timeseries_coords.append(newcoordinates)

timeseries_coords[0]
    
    



In [ ]:
reload(td)

In [ ]:
testdisplay = td.twoDimensionalPointMapper(full_data_normal[50], width=1024, height=800)
testdisplay.run()

In [ ]:
animation = td.twoDimensionalPointAnimator(full_data_circular, width=1024, height=800, frames_per_second=24, point_radius=1)
animation.animate()

In [ ]:
#create_circular_points
spectrogram_table

def convert_row_to_circle(point_list, radius_at_0=10):
    newpoint_list = point_list.copy()
    angle_step = 360/len(point_list)
    radius = radius_at_0
    for i in range(len(point_list)):
        angle = i * angle_step
        adjusted_radius = radius * (point_list[i]/20)
        adjusted_radius = max(adjusted_radius, radius_at_0)
        x = adjusted_radius * np.cos(np.radians(angle))
        y = adjusted_radius * np.sin(np.radians(angle))
        newpoint_list[i] = (x, y)
    return newpoint_list
  
full_data_circular = []
for columnnum in spectrogram_table.columns:
    all_data_for_time = list(spectrogram_table[columnnum])
    new_data = convert_row_to_circle(all_data_for_time)
    full_data_circular.append(new_data) 

full_data_circular


full_data_normal = []
for columnnum in spectrogram_table.columns:
    all_data_for_time = list(spectrogram_table[columnnum])
    new_data = []
    for i in range(len(all_data_for_time)):
        x = i
        y = all_data_for_time[i]
        new_data.append((x, y))
    full_data_normal.append(new_data)
#full_data_normal

### Create svg from data

### Check svg data

In [ ]:
def applying_function(*args, **kwargs):
    series = args[0]
    min_freq = series.name
    print()

    

def create_svg(*args, svg_dimensions = (800, 600), **kwargs):
    series = args[0]
    min_freq = series.name
    current_max = series.max()
    current_min = series.min()
    current_height = series[652]
    real_height = svg_dimensions[1] - current_height
    print(f'This is the series {min_freq} max: {current_max} and min:{current_min} and 652 value:{real_height}\n')
    #Start drawing svg 
    # step 1: Create points
    # Find equidistant points for time between 0 and svg_dimensions[0]
    # Map amplitude to y coordinate between 0 and svg_dimensions[1] (weighted like how blender does map from)
    


print(f'max = {spectrogram_table.max(axis=1).max()}, min = {spectrogram_table.min(axis=1).max()}')
lowest = spectrogram_table.min(axis=1).max()
highest = spectrogram_table.max(axis=1).max()
svg_size = (1024, highest+50)
smaller_table = spectrogram_table[(range(1214,1220))]

# Find the cell index (column) of the max value in the DataFrame
max_idx = spectrogram_table.stack().idxmax()
print(f'Max value is at row {max_idx[0]}, column {max_idx[1]}')
print(f'spectrogram table shape: {spectrogram_table.shape}')
spectrogram_table[304][10]
# Find frame to time in minutes and seconds
frame = 304
frame_time_seconds = times[frame]  # times is a list of time (in seconds) for each frame
minutes = int(frame_time_seconds // 60)
seconds = frame_time_seconds % 60
print(f"Frame {frame} occurs at {minutes} min {seconds:.2f} sec (={frame_time_seconds:.2f} sec)")
#spectrogram_table.apply(applying_function, axis=1, svg_dimensions=svg_size, max_amplitude=highest, min_amplitude=lowest)

# REALLY CREATE SVG UNDERNEATH

In [ ]:
def build_path(points, frame, svg_size=(1024, 800)):
    #for starting at top
    #path_string = f'<path id="dframe{frame}" d="M 0 0 '
    # for starting at bottom
    path_string = f'<path id="uframe{frame}" d="M 0 {int(svg_size[1])} '
    for point in points:
        x = point[0]
        y = int(point[1])
        path_string += f' L {x} {y}'
    #for starting at top
    #path_string += f' L {svg_size[0]} 0" stroke="none" stroke-width="1" fill="none"/>\n'
    #for strting at bottom
    path_string += f' L {svg_size[0]} {int(svg_size[1])}" stroke="none" stroke-width="1" fill="none"/>\n'
    return path_string

def build_svg(data_table, svg_size = (1024, 800)):
    final_number = 100000
    svg_string = ''
    svg_header = f'<svg height="{int(svg_size[1])}" width="{svg_size[0]}" xmlns="http://www.w3.org/2000/svg">\n' 
    svg_string += svg_header
    #to start at top
    #svg_starter = f'<path id="dstartframe" d="M 0 0 L {svg_size[0]} 0" stroke="red" stroke-width="1" fill="none"/>\n'
    #to start at bottom left
    svg_starter = f'<path id="ustartframe" d="M 0 {int(svg_size[1])} L {svg_size[0]} {int(svg_size[1])}" stroke="red" stroke-width="1" fill="none"/>\n'
    svg_string += svg_starter
    #svg_path_template = <path id="lineAB" d="M 100 350 l 150 -300" stroke="red" stroke-width="4"/>
    for columnnum in data_table.columns:
        columndata = data_table[columnnum]
        #print(f'Processing column: {columnnum}, data =')
        points = []
        for i in range(len(columndata)):
            value = columndata[i]
            true_value = svg_size[1] - value
            #for starting at top
            #coordinate = (i, value)
            #for starting at bottom
            coordinate = (i, true_value)
            #print(f'  coordinate: {coordinate}')
            points.append(coordinate)
        new_path = build_path(points, columnnum, svg_size=svg_size)
        svg_string += new_path
        final_number = columnnum
    #if starting from top
    #svg_starter = f'<path id="dframe{final_number+1}" d="M 0 0 L {svg_size[0]} 0" stroke="none" stroke-width="1" fill="none"/>\n'
    #if starting from bottom
    svg_string += f'<path id="uframe{final_number+1}" d="M 0 {int(svg_size[1])} L {svg_size[0]} {int(svg_size[1])}" stroke="none" stroke-width="1" fill="none"/>\n'
    svg_string += '</svg>'

    return svg_string


print(f'max = {spectrogram_table.max(axis=1).max()}, min = {spectrogram_table.min(axis=1).max()}')
lowest = spectrogram_table.min(axis=1).max()

highest = spectrogram_table.max(axis=1).max()
svg_size = (spectrogram_table.shape[0], highest+50)


svg = ''
height = int(svg_size[1])
highest
svgheader = f'<svg height="{height}" width="{svg_size[0]}" xmlns="http://www.w3.org/2000/svg">\n'
print(svgheader)
svg = build_svg(spectrogram_table, svg_size=svg_size)
print(svg)

#smaller_table.apply(create_svg, axis=0, svg_dimensions=svg_size, max_amplitude=highest, min_amplitude=lowest)

## output svg

In [ ]:
with open("./svgs/up_andre300.svg", "w") as f:
    f.write(svg)